# E11 FDT 违背与 Harada–Sasa 耗散闭环

本实验只验证一个结论：位置谱差本身不是功率，带权积分

$$
\dot Q_{\mathrm{probe}}=\gamma\int\frac{d\omega}{2\pi}\,\omega^2\Delta_x(\omega)
$$

才等于探针向平衡热浴的耗散率。对于加性 OU 活性力，解析结果是

$$
\dot Q_{\mathrm{probe}}=\frac{A}{2\tau_A(\gamma+k\tau_A)}.
$$

In [ ]:
import os
import sys

curr = os.path.abspath('')
while curr != os.path.dirname(curr):
    if 'statphys_urban_learning' in os.listdir(curr):
        target = os.path.join(curr, 'statphys_urban_learning')
        if target not in sys.path:
            sys.path.insert(0, target)
        break
    curr = os.path.dirname(curr)

import numpy as np
import matplotlib.pyplot as plt

from exercises.src.active_fdt import (
    analytic_probe_dissipation,
    equilibrium_fdt_position_spectrum,
    fdt_violation_position,
    integrate_even_spectrum,
    position_spectrum,
    probe_dissipation_density,
    simulate_active_ou_probe,
)

## 1. 分开画出涨落、响应预测与谱差

蓝线由温度和响应函数给出；橙线是实际位置涨落谱。只有活性强度 $A=0$ 时，两条线才逐频重合。

In [ ]:
k = 1.4
gamma = 0.9
kbt = 1.0
A = 2.2
tau_A = 0.7

omega = np.logspace(-3, 3, 3000)
c_total = position_spectrum(
    omega, k=k, gamma=gamma, kbt=kbt, active_strength=A, active_tau=tau_A
)
c_eq = equilibrium_fdt_position_spectrum(omega, k=k, gamma=gamma, kbt=kbt)
delta = fdt_violation_position(
    omega, k=k, gamma=gamma, active_strength=A, active_tau=tau_A
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].loglog(omega, c_eq, label='FDT prediction', lw=2)
axes[0].loglog(omega, c_total, label='measured total spectrum', lw=2)
axes[0].set(xlabel='omega', ylabel='C_xx(omega)', title='Fluctuation versus response prediction')
axes[0].legend()
axes[0].grid(True, which='both', alpha=0.2)

axes[1].loglog(omega, delta, color='#10b981', lw=2)
axes[1].set(xlabel='omega', ylabel='Delta_x(omega)', title='Position-spectrum FDT violation')
axes[1].grid(True, which='both', alpha=0.2)
plt.tight_layout()
plt.show()

## 2. 数值积分必须与闭式耗散率一致

这里在非负频率上积分偶函数，因此使用 $\int_0^\infty q(\omega)d\omega/\pi$。扩大截止频率，检查结果是否收敛，而不是只报告一次积分。

In [ ]:
analytic = analytic_probe_dissipation(
    k=k, gamma=gamma, active_strength=A, active_tau=tau_A
)

for omega_max in [10, 100, 1_000, 10_000]:
    w = np.concatenate(([0.0], np.logspace(-6, np.log10(omega_max), 50_000)))
    q = probe_dissipation_density(
        w, k=k, gamma=gamma, active_strength=A, active_tau=tau_A
    )
    numerical = integrate_even_spectrum(w, q)
    print(f'omega_max={omega_max:>6}: numerical={numerical:.8f}, error={numerical-analytic:+.3e}')

print('analytic:', analytic)

## 3. 轨迹层检查活性力的相关尺度

模拟器同时返回探针位置和隐藏的 OU 活性力。后者的稳态方差应为 $A/(2\tau_A)$。真实实验通常看不到这条隐藏力，这正是“探针可见耗散”不等于“完整系统耗散”的原因。

In [ ]:
trace = simulate_active_ou_probe(
    k=k, gamma=gamma, kbt=kbt, active_strength=A, active_tau=tau_A,
    dt=1e-3, n_steps=250_000, seed=11,
)
burn = 20_000
measured_active_var = np.var(trace.active_force[burn:])
expected_active_var = A / (2 * tau_A)
print('active-force variance (measured):', measured_active_var)
print('active-force variance (theory)  :', expected_active_var)

window = slice(burn, burn + 5000)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(trace.t[window], trace.x[window], lw=1)
axes[0].set_ylabel('x(t)')
axes[1].plot(trace.t[window], trace.active_force[window], color='#f97316', lw=1)
axes[1].set(xlabel='t', ylabel='xi_A(t)')
plt.tight_layout()
plt.show()

## 讨论

1. 令 $A=0$，确认总位置谱与 FDT 预测逐频重合。
2. 分别增大 $k$ 与 $\tau_A$，解释耗散率为什么下降。
3. 为什么直接积分 $\Delta_x(\omega)$ 不能得到功率？请用量纲回答。
4. 当 $\tau_A\to0$ 时过阻尼结果为什么发散？这暴露了哪个被省略的短时尺度？
5. 实验只记录 $x(t)$ 时，哪些活性耗散通道永远不会出现在本积分中？